In [112]:
# importacion de librerias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [113]:
#lectura de datasets
df_victimas = pd.read_csv('Estatal-Víctimas-2015-2025_may2026.csv', encoding='latin1')

df_inegi = pd.read_excel('INEGI_exporta_17_7_2026_15_49_30_limpio.xlsx')

In [114]:
#copia de los dataset
df_copia = df_victimas.copy()
df_copy = df_inegi.copy()


In [115]:
#renombrando columans del df_inegi
df_inegi = df_inegi.rename(columns={
    ' Total': 'Entidad',
    ' Total.1': 'Poblacion'
})

print(df_inegi)

    Unnamed: 0                           Entidad  Poblacion   Hombres  \
0            1                    Aguascalientes    1425607    696683   
1            2                   Baja California    3769020   1900589   
2            3               Baja California Sur     798447    405879   
3            4                          Campeche     928363    456939   
4            5              Coahuila de Zaragoza    3146771   1563669   
5            6                            Colima     731391    360622   
6            7                           Chiapas    5543828   2705947   
7            8                         Chihuahua    3741869   1853822   
8            9                  Ciudad de México    9209944   4404927   
9           10                           Durango    1832650    904866   
10          11                        Guanajuato    6166934   2996454   
11          12                          Guerrero    3540685   1700612   
12          13                           Hidalgo   

In [116]:
#LIMPIANDO DETALLES DE LOS NOMBRES DE LAS ENTIDADES DEL df_entidad: #aqui se elimino un espacio que tenia este estado
df_inegi["Entidad"] = df_inegi["Entidad"].str.strip()

In [117]:
#construyendo el diccionario

poblacion_estados = {'Aguascalientes':1425607,'Baja California':3769020,'Baja California Sur':798447,'Campeche':928363,
                     'Coahuila de Zaragoza':3146771,'Colima':731391,'Chiapas':5543828,'Chihuahua':3741869,'Ciudad de México':9209944,
                     'Durango':1832650,'Guanajuato':6166934,'Guerrero':3540685,'Hidalgo':3082841,'Jalisco':8348151,'México':16992418,
                     'Michoacán de Ocampo':4748846,'Morelos':1971520,'Nayarit':1235456,'Nuevo León':5784442,'Oaxaca':4132148,'Puebla':6583278,
                     'Querétaro':2368467,'Quintana Roo':1857985,'San Luis Potosí':2822255,'Sinaloa':3026943,'Sonora':2944840,'Tabasco':2402598,
                     'Tamaulipas':3527735,'Tlaxcala':1342977,'Veracruz de Ignacio de la Llave':8062579,'Yucatán':2320898,'Zacatecas':1622138}

# !!!!!!!!!!!!!!!!!!!!! PREGUNTAS NUEVAS QUE MANDO ALEJANDRO
# ------- Reformulacion de pregunta #
## para contestarla bien primero hay que:
## Normalizar los datos calculando la tasa de delitos por cada 100,000 habitantes
## primero busca los datos de población por estado el INEGI para los años de su estudio. La formula que deben aplicar para cada estado/estado
## Tasa = (total de delitos / poblacion del estado) x (100,000)

# NOTA:
## TODAS SUS PRUEBAS ESTADISTICAS (ANOVA, REGRESION, ETC.) DEBEN HACERSE CON ESTA TASA NORMALIZADA, NO CON NUMEROS ABSOLUTOS
# 3- ¿La proporción o el promedio de victimas masculinas vs femeninas difiere significativamente? (pruebas estadisticas de comparacion de dos grupos)

In [118]:
meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
         'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

#AGRUPANDO DATOS
resultado = (
    df_copia
    .groupby(["Entidad", "Año", "Sexo"])[meses]
    .sum()
    .reset_index()
)

#print(resultado)





#  suma cada columna de enero a diciembre de cada estado
resultado["Total"] = resultado[meses].sum(axis=1)
print(resultado['Total'])

0        248
1         70
2       3586
3        262
4         74
        ... 
1051    1746
1052      79
1053    2928
1054    1553
1055      65
Name: Total, Length: 1056, dtype: int64


In [119]:

#lo que hace .map() es: busca cada dato en el diccionario, ejemplo:
#para 'puebla' busca 'poblacion_estados["Puebla"]' y encuentra: '6583278' y agrega el numero en la columna 'Poblacion'

resultado["Poblacion"] = resultado["Entidad"].map(poblacion_estados)


#excluyendo los datos de 'no identificado' de la columna 'Sexo'
df_filtrado = df_copia[df_copia["Sexo"] != "No identificado"]
print(df_filtrado['Sexo'])


0         Mujer
1         Mujer
2         Mujer
3         Mujer
4         Mujer
          ...  
80923    Hombre
80924    Hombre
80925    Hombre
80926    Hombre
80927    Hombre
Name: Sexo, Length: 69696, dtype: object


In [120]:
resultado = (
    df_filtrado
    .groupby(["Entidad", "Sexo"])[meses]
    .sum()
    .reset_index()
)

In [121]:
resultado["Total"] = resultado[meses].sum(axis=1)

In [122]:
resultado["Poblacion"] = resultado["Entidad"].map(poblacion_estados)

In [123]:
resultado["Tasa_pregunta3"] = (
    resultado["Total"] / resultado["Poblacion"]
) * 100000

In [124]:
hombres = resultado[resultado["Sexo"] == "Hombre"]["Tasa_pregunta3"]

In [125]:
mujeres = resultado[resultado['Sexo'] == 'Mujer']['Tasa_pregunta3']

In [126]:
df_filtrado["Sexo"].unique()

array(['Mujer', 'Hombre'], dtype=object)

Paso 6. Hacer la prueba estadística

Aquí es donde entra la inferencia.

Generalmente el flujo es:

Revisar si ambas muestras siguen una distribución aproximadamente normal.
Si sí, usar una prueba t para dos muestras independientes.
Si no, usar una prueba no paramétrica como Mann–Whitney U.

Eso coincide con lo que la profesora les comentó en correos anteriores.

In [127]:
resultado["Sexo"].value_counts()

Sexo
Hombre    32
Mujer     32
Name: count, dtype: int64